ModuleNotFoundError: No module named 'yfinance'

In [2]:
%pip install yfinance

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------------------- ------------- 1.3/2.0 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 6.9 MB/s  0:00:00

   -------- ------------------------------- 1/5 [websockets]
   ---------------- ----------------------- 2/5 [peewee]
   ------------------------ --------------- 3/5 [curl_cffi]
   ------------------------ --------------- 3/5 [curl_cffi]
   -------------------------------- ------- 4/5 [yfinance]
   ---------------------------------------- 5/5 [yfinance]

Note: you may need to restart the kernel to use updated packages.


In [6]:
import yfinance as yf
import pandas as pd
from pathlib import Path
from datetime import date

def pull_prices(tickers: dict, start: str, out_dir: Path, lineage: str):
    """Pull full available daily OHLCV history (start date through today).
    Date-window and era filtering happen downstream in Power Query, not here."""
    out_dir.mkdir(parents=True, exist_ok=True)
    end = date.today().isoformat()
    for ticker, name in tickers.items():
        print(f"Pulling {ticker} ({name})...")
        df = yf.download(ticker, start=start, end=end, auto_adjust=False)
        if df.empty:
            print(f"  WARNING: no data returned for {ticker}")
            continue
        df = df.reset_index()
        df["Ticker"] = ticker
        df["Lineage"] = lineage
        out_path = out_dir / f"{ticker}.csv"
        df.to_csv(out_path, index=False)
        print(f"  saved {len(df)} rows to {out_path} "
              f"(spans {df['Date'].min().date()} to {df['Date'].max().date()})")

TANKER_TICKERS = {
    "CMBT.BR": "Euronav / CMB.TECH (Brussels listing, full history)",
    "FRO":  "Frontline",
    "DHT":  "DHT Holdings",
    "TNK":  "Teekay Tankers",
    "STNG": "Scorpio Tankers",
}
pull_prices(TANKER_TICKERS, start="2004-01-01", out_dir=Path("../raw/tanker"), lineage="Tanker")

Pulling CMBT.BR (Euronav / CMB.TECH (Brussels listing, full history))...


[*********************100%***********************]  1 of 1 completed


  saved 5575 rows to ..\raw\tanker\CMBT.BR.csv (spans 2004-12-01 to 2026-09-08)
Pulling FRO (Frontline)...


[*********************100%***********************]  1 of 1 completed


  saved 5706 rows to ..\raw\tanker\FRO.csv (spans 2004-01-02 to 2026-09-08)
Pulling DHT (DHT Holdings)...


...

  saved 5257 rows to ..\raw\tanker\DHT.csv (spans 2005-10-13 to 2026-09-08)
Pulling TNK (Teekay Tankers)...


[*********************100%***********************]  1 of 1 completed


  saved 4712 rows to ..\raw\tanker\TNK.csv (spans 2007-12-13 to 2026-09-08)
Pulling STNG (Scorpio Tankers)...


[*********************100%***********************]  1 of 1 completed

  saved 4135 rows to ..\raw\tanker\STNG.csv (spans 2010-03-31 to 2026-09-08)
